# 10 - Noise Assignment

Following source `09_noise.ipynb`:
1. Identify noise tessellation cells (morphotope label ending in `_-1`)
2. Group noise by spatial contiguity (connected components)
3. Compute median characters per noise group
4. `KNeighborsClassifier(k=5)` trained on cluster centroids
5. Assign noise groups to nearest cluster

In [1]:
import os
if os.path.basename(os.getcwd()).startswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))
print("Working dir:", os.getcwd())

import sys
sys.path.insert(0, "scripts")
from pipeline_config import load_city_config
load_city_config()

import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from libpysal.graph import read_parquet
from pathlib import Path

OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "outputs")
MORPH_DIR = Path(OUTPUT_ROOT) / "06_morphotopes"
PRIMARY_CHARS_DIR = Path(OUTPUT_ROOT) / "05_primary_chars"
GRAPH_DIR = Path(OUTPUT_ROOT) / "03_graphs"
HIER_DIR = Path(OUTPUT_ROOT) / "09_hierarchy"

Working dir: C:\urban-form\urban_taxonomy
[pipeline_config] City config: config\canada_cma_ca_nochm.env


In [2]:
morph_labels = pd.read_parquet(
    MORPH_DIR / "tessellation_labels_postprocessed.parquet"
)["morphotope_label"]
X_train = pd.read_parquet(PRIMARY_CHARS_DIR / "primary_chars.parquet")
tess_graph = read_parquet(GRAPH_DIR / "tessellation_graph.parquet")
cdata = pd.read_parquet(HIER_DIR / "cdata.parquet")
cluster_mapping = pd.read_parquet(HIER_DIR / "cluster_mapping.parquet")
tess_clusters = pd.read_parquet(HIER_DIR / "tessellation_clusters.parquet")

# Identify noise
is_noise = morph_labels.str.endswith('_-1')
noise_idx = morph_labels[is_noise].index
print(f"Noise cells: {len(noise_idx):,} / {len(morph_labels):,} ({100*len(noise_idx)/len(morph_labels):.1f}%)")

Noise cells: 1,011 / 6,668 (15.2%)


In [3]:
# Group noise by spatial contiguity
noise_graph = tess_graph.subgraph(noise_idx)
noise_components = noise_graph.component_labels

# Compute median chars per noise group
noise_chars = X_train.loc[noise_idx]
noise_medians = noise_chars.groupby(noise_components.values).median()

# Approximate limAre (sum of top-30 building areas per noise group)
if 'sdbAre' in noise_chars.columns:
    limAre_approx = noise_chars['sdbAre'].groupby(noise_components.values).apply(
        lambda x: x.nlargest(min(30, len(x))).sum()
    )
    noise_medians['limAre'] = limAre_approx

print(f"{len(noise_medians)} noise groups")

71 noise groups


In [4]:
# Align columns between noise medians and cluster centroids
# Source drops limLPS before noise assignment
cdata_for_noise = cdata.drop(columns=['limLPS'], errors='ignore')
common_cols = noise_medians.columns.intersection(cdata_for_noise.columns)
noise_feat = noise_medians[common_cols].fillna(0)
cluster_feat = cdata_for_noise[common_cols].fillna(0)

# Scale together
scaler = StandardScaler()
all_feat = pd.concat([cluster_feat, noise_feat])
all_scaled = scaler.fit_transform(all_feat)
cluster_scaled = all_scaled[:len(cluster_feat)]
noise_scaled = all_scaled[len(cluster_feat):]

# KNN classifier (source: k=5)
finest_labels = cluster_mapping['level_1']
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(cluster_scaled, finest_labels.values)
noise_predictions = knn.predict(noise_scaled)

# Map back: noise component -> predicted cluster -> individual cells
comp_to_cluster = dict(zip(noise_medians.index, noise_predictions))
noise_cell_clusters = noise_components.map(comp_to_cluster)

print(f"Assigned {len(noise_cell_clusters):,} noise cells to {len(set(noise_predictions))} clusters")

Assigned 1,011 noise cells to 2 clusters


In [5]:
# Update tessellation clusters
tess_final = tess_clusters.copy()
tess_final['final_without_noise'] = tess_final['final']  # preserve original

# Fill in noise cells
for idx in noise_cell_clusters.index:
    if idx in tess_final.index:
        tess_final.loc[idx, 'final'] = noise_cell_clusters.loc[idx]
        # Also propagate to all hierarchy levels
        assigned_morph = cdata.index[finest_labels == noise_cell_clusters.loc[idx]][0]
        for col in cluster_mapping.columns:
            tess_final.loc[idx, col] = cluster_mapping.loc[assigned_morph, col]

tess_final['initially_noise'] = False
tess_final.loc[noise_idx.intersection(tess_final.index), 'initially_noise'] = True

tess_final.to_parquet(HIER_DIR / "tessellation_clusters_final.parquet")
print(f"Final clusters saved: {tess_final['final'].notna().sum():,} assigned, {tess_final['final'].isna().sum():,} unassigned")
print(f"Saved to {HIER_DIR / 'tessellation_clusters_final.parquet'}")

Final clusters saved: 6,668 assigned, 0 unassigned
Saved to outputs_canada_cma_ca_nochm\regions\995\09_hierarchy\tessellation_clusters_final.parquet
